# Rossmann Sales Forecasting

## Project Overview

**Project Name:** Rossmann Sales Forecasting

**Aim of the Project:**  
Develop a machine learning-based time series forecasting solution to predict daily sales for Rossmann stores using historical sales data, store information, promotions, holidays, and other relevant features.

**Notebook 1 - Data Understanding:**
A brief introduction and basic understanding of the dataset before moving onto next steps

**Dataset:**  
Rossmann Store Sales dataset from Kaggle.

**Target Variable:**  
`Sales` — daily sales/turnover of a Rossmann store.

## 1. Notebook setup and data loading

Primary inspection and validation of data files

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CURRENT_FOLDER = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_FOLDER.parent
    if CURRENT_FOLDER.name.lower() == "notebooks"
    else CURRENT_FOLDER
)

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"
STORE_PATH = PROJECT_ROOT / "data" / "raw" / "store.csv"
TEST_PATH = PROJECT_ROOT / "data" / "raw" / "test.csv"

for file in [ TRAIN_PATH, STORE_PATH, TEST_PATH]:
    if file.exists():
        print(
            f'✅ Found {file.name} |'
            f'Path: {file} | file size: {file.stat().st_size / (1024 * 1024):.2f} MB'
            )
    else:
        raise FileNotFoundError(f"❌ File not found: {file}")


✅ Found train.csv |Path: D:\Projects\Rossmann_Sales_Forecasting\data\raw\train.csv | file size: 36.29 MB
✅ Found store.csv |Path: D:\Projects\Rossmann_Sales_Forecasting\data\raw\store.csv | file size: 0.04 MB
✅ Found test.csv |Path: D:\Projects\Rossmann_Sales_Forecasting\data\raw\test.csv | file size: 1.36 MB


### 2. Load the datasets

The Rossmann dataset is split into several CSV files:

- `train.csv` — historical data including actual sales
- `test.csv` — data where sales are not provided ( target period )
- `store.csv` — information about each store ( store metadata )

**Note on Data Types:** The StateHoliday column contains mixed data types (a mix of strings like 'a', 'b', 'c' and numerical zeros 0). To prevent pandas from inferring mixed types and throwing memory/parsing warnings, explicitly force it to load as a string right at ingestion.

In [ ]:

train = pd.read_csv(TRAIN_PATH, dtype={"StateHoliday": "string"})
test = pd.read_csv(TEST_PATH, dtype={"StateHoliday": "string"})
store = pd.read_csv(STORE_PATH)

print(f"Train unique values: {train['StateHoliday'].unique().tolist()}")
print(f"Test unique values: {test['StateHoliday'].unique().tolist()}")


Train unique values: ['0', 'a', 'b', 'c']
Test unique values: ['0', 'a']


### 3. Size of datasets


In [ ]:

display(pd.DataFrame({
    'Rows': [train.shape[0], test.shape[0], store.shape[0]],
    'Columns': [train.shape[1], test.shape[1], store.shape[1]]
}, index=['Train', 'Test', 'Store']))



,Rows,Columns
Train,1017209,9
Test,41088,8
Store,1115,10


### 5. First 5 rows


In [4]:
train.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


### 5. Row context

For the Rossmann daily data, one row represents the information for **one store on one particular date**.

For example:

```text
Store = 25
Date = 2015-07-31
Sales = 8500
Promo = 1
...
```

The data contains repeated observations for stores across time.

### 6. Columns

Verifying feature consistency and data types prior to executing data merges and feature engineering.

In [5]:
display(pd.DataFrame(train.columns.tolist(), columns= ['Column']))
display(pd.DataFrame(store.columns.tolist(), columns= ['Column']))
display(pd.DataFrame(test.columns.tolist(), columns= ['Column']))

,Column
0,Store
1,DayOfWeek
2,Date
3,Sales
4,Customers
5,Open
6,Promo
7,StateHoliday
8,SchoolHoliday


,Column
0,Store
1,StoreType
2,Assortment
3,CompetitionDistance
4,CompetitionOpenSinceMonth
5,CompetitionOpenSinceYear
6,Promo2
7,Promo2SinceWeek
8,Promo2SinceYear
9,PromoInterval


,Column
0,Id
1,Store
2,DayOfWeek
3,Date
4,Open
5,Promo
6,StateHoliday
7,SchoolHoliday


### 7. Target of the dataset

Verifying the target feature `Sales` only appears in training dataset.

In [6]:
print("Is Sales in train?", "Sales" in train.columns)
print("Is Sales in test?", "Sales" in test.columns)

Is Sales in train? True
Is Sales in test? False


### 8. Column data types


In [7]:
print("Train data:")
display(train.dtypes.reset_index().set_axis(['Column', 'Data Type'], axis=1))

print("Store metadata:")
display(store.dtypes.reset_index().set_axis(['Column', 'Data Type'], axis=1))


Train data:


,Column,Data Type
0,Store,int64
1,DayOfWeek,int64
2,Date,str
3,Sales,int64
4,Customers,int64
5,Open,int64
6,Promo,int64
7,StateHoliday,string
8,SchoolHoliday,int64


Store metadata:


,Column,Data Type
0,Store,int64
1,StoreType,str
2,Assortment,str
3,CompetitionDistance,float64
4,CompetitionOpenSinceMonth,float64
5,CompetitionOpenSinceYear,float64
6,Promo2,int64
7,Promo2SinceWeek,float64
8,Promo2SinceYear,float64
9,PromoInterval,str


### 9. Check the date column


In [8]:
train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])

print(train["Date"].dtype)
print(test["Date"].dtype)

datetime64[us]
datetime64[us]


### 10. Find the date range

Finding data range in order to know how much historical time the dataset covers. An important information for later when splitting earlier data for training and later data for validation.

In [9]:
print("Training start date:", train["Date"].min())
print("Training end date:", train["Date"].max())
print("Test start date:", test["Date"].min())
print("Test end date:", test["Date"].max())

Training start date: 2013-01-01 00:00:00
Training end date: 2015-07-31 00:00:00
Test start date: 2015-08-01 00:00:00
Test end date: 2015-09-17 00:00:00


### 11. Total number of stores


In [10]:
print("Unique stores in train:", train["Store"].nunique())
print("Unique stores in test:", test["Store"].nunique())
print("Stores in store.csv:", store["Store"].nunique())

Unique stores in train: 1115
Unique stores in test: 856
Stores in store.csv: 1115


### 12. Missing values check


In [11]:
print("Training data")
display(train.isna().sum().reset_index().rename(columns={'index': 'Column', 0: 'Missing Values'}))

print("Store metadata")
display(store.isna().sum().reset_index().rename(columns={'index': 'Column', 0: 'Missing Values'}))



Training data


,Column,Missing Values
0,Store,0
1,DayOfWeek,0
2,Date,0
3,Sales,0
4,Customers,0
5,Open,0
6,Promo,0
7,StateHoliday,0
8,SchoolHoliday,0


Store metadata


,Column,Missing Values
0,Store,0
1,StoreType,0
2,Assortment,0
3,CompetitionDistance,3
4,CompetitionOpenSinceMonth,354
5,CompetitionOpenSinceYear,354
6,Promo2,0
7,Promo2SinceWeek,544
8,Promo2SinceYear,544
9,PromoInterval,544


### 13. Check for duplicate rows


In [12]:
print("Duplicate rows in train:", train.duplicated().sum())
print("Duplicate rows in test:", test.duplicated().sum())
print("Duplicate rows in store:", store.duplicated().sum())

Duplicate rows in train: 0
Duplicate rows in test: 0
Duplicate rows in store: 0


### 14. Basic statistics for numerical columns

Quick numerical summary such as the average, minimum and maximum values. This gives an initial feel for variables such as `Sales` and `Customers`.

In [13]:
train.describe()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,SchoolHoliday
count,1.017209e+06,1.017209e+06,1017209,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06
mean,5.584297e+02,3.998341e+00,2014-04-11 01:30:42.846062,5.773819e+03,6.331459e+02,8.301067e-01,3.815145e-01,1.786467e-01
min,1.000000e+00,1.000000e+00,2013-01-01 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.800000e+02,2.000000e+00,2013-08-17 00:00:00,3.727000e+03,4.050000e+02,1.000000e+00,0.000000e+00,0.000000e+00
50%,5.580000e+02,4.000000e+00,2014-04-02 00:00:00,5.744000e+03,6.090000e+02,1.000000e+00,0.000000e+00,0.000000e+00
75%,8.380000e+02,6.000000e+00,2014-12-12 00:00:00,7.856000e+03,8.370000e+02,1.000000e+00,1.000000e+00,0.000000e+00
max,1.115000e+03,7.000000e+00,2015-07-31 00:00:00,4.155100e+04,7.388000e+03,1.000000e+00,1.000000e+00,1.000000e+00
std,3.219087e+02,1.997391e+00,NaN,3.849926e+03,4.644117e+02,3.755392e-01,4.857586e-01,3.830564e-01


### 15. Understand important categorical columns

Understanding important categorical columns . Examples include `StateHoliday`, `StoreType`, and `Assortment`.

In [14]:
print("Train data:")
print("StateHoliday values:")
display(pd.DataFrame(train["StateHoliday"].value_counts(dropna=False)).reset_index())


print("Store metadata:")
print("\nStoreType values:")
display(pd.DataFrame(store["StoreType"].value_counts(dropna=False)).reset_index())

print("\nAssortment values:")
display(pd.DataFrame(store["Assortment"].value_counts(dropna=False)).reset_index())

print("\nPromomInterval values:")
display(pd.DataFrame(store["PromoInterval"].value_counts(dropna=False)).reset_index())

Train data:
StateHoliday values:


,StateHoliday,count
0,0,986159
1,a,20260
2,b,6690
3,c,4100


Store metadata:

StoreType values:


,StoreType,count
0,a,602
1,d,348
2,c,148
3,b,17



Assortment values:


,Assortment,count
0,a,593
1,c,513
2,b,9



PromomInterval values:


,PromoInterval,count
0,NaN,544
1,"Jan,Apr,Jul,Oct",335
2,"Feb,May,Aug,Nov",130
3,"Mar,Jun,Sept,Dec",106


### 16. Understand the store table

`train.csv` contains information that changes by date, such as sales and promotions. `store.csv` contains information about the stores themselves, such as store type and competition distance. Both tables have a `Store` column, so that it can be used later to combine the information.

In [15]:
store.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


### 17. Primary Data Inspection Summary

### Key points

- `train.csv` contains historical daily observations and the target `Sales`.
- `test.csv` contains observations where `Sales` is not provided.
- `store.csv` contains information about each store.
- `Store` connects daily data with store information.
- One row in the daily data represents one store on one date.
- `Date` is central to our forecasting problem.
- `Sales` is the value we want to predict.
- Missing values and duplicates should be investigated before modelling.
- We must avoid using information that would not be available when making a future prediction.

## Next step

Exploratory Data Analysis to see raw data trends and patterns.